In [ ]:
import uuid
import time
import hashlib
from datetime import datetime, timedelta, timezone
from typing import Optional

# IDENTITY STORE

In [ ]:
_active_tokens: dict[str, dict] = {}

In [ ]:
def _mint_token(scope: list[str], ttl_seconds: int = 30) -> str:
    """Create a short-lived scoped token and register it."""
    token_id = str(uuid.uuid4())
    _active_tokens[token_id] = {
        "scope": scope,
        "expires_at": datetime.now(timezone.utc) + timedelta(seconds=ttl_seconds),
        "used": False,
    }
    return token_id


def _revoke_token(token_id: str) -> None:
    """Destroy a token — after this, it cannot be used."""
    _active_tokens.pop(token_id, None)


def _validate_token(token_id: str, action: str) -> bool:
    """Check if a token is valid and permits the requested action."""
    record = _active_tokens.get(token_id)
    if not record:
        return False  # already revoked or never existed
    if datetime.now(timezone.utc) > record["expires_at"]:
        _revoke_token(token_id)
        return False  # expired
    if action not in record["scope"]:
        return False  # out of scope
    return True


# SIMULATED RESOURCES

In [ ]:
class SecureResource:
    """A resource that requires a valid scoped token to access."""

    _data = {
        "read_file": " Contents of report.txt: Q3 revenue grew 12%.",
        "send_email": " Email sent to team@company.com.",
        "query_db":   "  DB result: 42 active users today.",
    }

    def access(self, token_id: str, action: str) -> str:
        if not _validate_token(token_id, action):
            raise PermissionError(
                f"[DENIED] Token '{token_id[:8]}…' cannot perform '{action}'. "
                "Token may be expired, revoked, or out of scope."
            )
        return self._data.get(action, "Action completed.")

# EPHEMERAL AGENT

In [ ]:
class EphemeralAgent:
    """
    A short-lived agent that:
      - Receives a scoped token (not the raw credentials)
      - Performs its task
      - Exposes NO internal state to the caller
      - Is destroyed after use
    """

    def __init__(self, agent_id: str, token_id: str, task: str):
        self._agent_id = agent_id
        self._token_id = token_id   # private — caller never sees this
        self._task = task
        self._memory: list[str] = []   # internal only, wiped on destroy
        self._alive = True

    def run(self, resource: SecureResource) -> str:
        if not self._alive:
            raise RuntimeError("Agent has already been destroyed.")

        print(f"  [Agent {self._agent_id[:8]}] Starting task: '{self._task}'")
        self._memory.append(f"Task received: {self._task}")

        # Perform work using the scoped token
        result = resource.access(self._token_id, self._task)
        self._memory.append(f"Result obtained: {result}")

        print(f"  [Agent {self._agent_id[:8]}] Task complete. Result obtained.")
        return result   # only the result is returned — not the token or memory

    def destroy(self) -> None:
        """Wipe all internal state and revoke credentials."""
        _revoke_token(self._token_id)
        self._memory.clear()
        self._token_id = "[REVOKED]"
        self._alive = False
        print(f"  [Agent {self._agent_id[:8]}] Destroyed. Token revoked. Memory wiped.")


# ORCHESTRATOR

In [ ]:
class Orchestrator:
    """
    Manages agent lifecycle:
      - Mints a scoped token for each task
      - Spawns an agent (agent never receives raw creds)
      - Destroys the agent after use
      - Returns only the result to the caller
    """

    def __init__(self):
        self._resource = SecureResource()

    def execute_task(self, task: str, allowed_scope: list[str]) -> str:
        print(f"\n[Orchestrator] Task received: '{task}'")

        # Step 1: Mint a scoped, short-lived token
        token_id = _mint_token(scope=allowed_scope, ttl_seconds=15)
        print(f"[Orchestrator] Minted token (scope={allowed_scope}, TTL=15s)")

        # Step 2: Spawn ephemeral agent — agent only gets token_id, not raw creds
        agent_id = str(uuid.uuid4())
        agent = EphemeralAgent(agent_id=agent_id, token_id=token_id, task=task)
        print(f"[Orchestrator] Spawned agent {agent_id[:8]}…")

        # Step 3: Run
        try:
            result = agent.run(self._resource)
        except PermissionError as e:
            result = str(e)
        finally:
            # Step 4: Always destroy the agent — even if an error occurred
            agent.destroy()

        print(f"[Orchestrator] Returning result to manager (no credentials shared).")
        return result


# MANAGER


In [ ]:
class Manager:
    """
    The manager:
      - Submits tasks with a defined scope
      - Receives ONLY the result
      - Has zero access to agent internals or credentials
    """

    def __init__(self):
        self._orchestrator = Orchestrator()

    def request(self, task: str, scope: list[str]) -> None:
        print(f"\n{'='*55}")
        print(f"[Manager] Requesting task '{task}' with scope {scope}")
        result = self._orchestrator.execute_task(task, allowed_scope=scope)
        print(f"[Manager]  Result received: {result}")
        print(f"{'='*55}")

# DEMO

In [ ]:
if __name__ == "__main__":
    print("\n EPHEMERAL AGENT IDENTITY PROTOTYPE\n")
    manager = Manager()

    # Task 1: Allowed — read_file is in scope
    manager.request(task="read_file", scope=["read_file"])

    # Task 2: Allowed — query_db is in scope
    manager.request(task="query_db", scope=["query_db", "read_file"])

    # Task 3: Denied — send_email is NOT in the granted scope
    manager.request(task="send_email", scope=["read_file"])

    # Task 4: Demonstrate token cannot be reused after agent is destroyed
    print(f"\n{'='*55}")
    print("[Demo] Trying to reuse a destroyed agent's token...")
    dead_token = _mint_token(scope=["read_file"], ttl_seconds=30)
    _revoke_token(dead_token)  # simulate destruction
    resource = SecureResource()
    try:
        resource.access(dead_token, "read_file")
    except PermissionError as e:
        print(f"[Demo]  Correctly blocked: {e}")
    print(f"{'='*55}\n")


 EPHEMERAL AGENT IDENTITY PROTOTYPE


[Manager] Requesting task 'read_file' with scope ['read_file']

[Orchestrator] Task received: 'read_file'
[Orchestrator] Minted token (scope=['read_file'], TTL=15s)
[Orchestrator] Spawned agent add8d9a8…
  [Agent add8d9a8] Starting task: 'read_file'
  [Agent add8d9a8] Task complete. Result obtained.
  [Agent add8d9a8] Destroyed. Token revoked. Memory wiped.
[Orchestrator] Returning result to manager (no credentials shared).
[Manager]  Result received:  Contents of report.txt: Q3 revenue grew 12%.

[Manager] Requesting task 'query_db' with scope ['query_db', 'read_file']

[Orchestrator] Task received: 'query_db'
[Orchestrator] Minted token (scope=['query_db', 'read_file'], TTL=15s)
[Orchestrator] Spawned agent 28070569…
  [Agent 28070569] Starting task: 'query_db'
  [Agent 28070569] Task complete. Result obtained.
  [Agent 28070569] Destroyed. Token revoked. Memory wiped.
[Orchestrator] Returning result to manager (no credentials shared).
[Manager]